<a href="https://colab.research.google.com/github/sukritghosh886-hub/-ai-model-reliability-lab./blob/main/AI_Model_Reliability_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


%pip -q install "fastapi>=0.110,<1.0" "uvicorn>=0.27,<1.0" "scikit-learn>=1.4,<2.0" "httpx>=0.26,<1.0" "pytest>=8,<9"
print("Installation completed")

Installation completed


In [2]:


import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)

# Load a small built-in benchmark; no manual download
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target,
    test_size=0.25,
    random_state=42,
    stratify=data.target
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42)
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)
confidence = probabilities.max(axis=1)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)
errors = np.flatnonzero(predictions != y_test)
high_confidence_errors = [
    int(i) for i in errors if confidence[i] >= 0.80
]

print("AI MODEL RELIABILITY LAB")
print("=" * 40)
print("Test samples:", len(y_test))
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))
print("\nIncorrect predictions:", len(errors))
print("Incorrect predictions with confidence >= 80%:",
      len(high_confidence_errors))
print("Brier score (lower is better):",
      round(brier_score_loss(y_test, probabilities[:, 1]), 4))
print("\nNote: educational benchmark only, not a clinical tool.")

AI MODEL RELIABILITY LAB
Test samples: 143
Accuracy:  0.9860
Precision: 0.9889
Recall:    0.9889
F1-score:  0.9889

Confusion matrix:
[[52  1]
 [ 1 89]]

Incorrect predictions: 2
Incorrect predictions with confidence >= 80%: 1
Brier score (lower is better): 0.0181

Note: educational benchmark only, not a clinical tool.


In [3]:


recommendations = []

if recall < 0.80:
    recommendations.append(
        "Investigate false negatives and test the decision threshold."
    )

if precision < 0.80:
    recommendations.append(
        "Investigate false positives and improve feature quality."
    )

if accuracy < 0.80:
    recommendations.append(
        "Review data quality and compare with a baseline model."
    )

if len(high_confidence_errors) > 0:
    recommendations.append(
        "Investigate incorrect predictions made with high confidence."
    )

if not recommendations:
    recommendations.append(
        "No basic metric warning triggered. Test on independent data."
    )

print("RELIABILITY REPORT")
print("-" * 40)
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")

print("\nImportant: high confidence is not automatically calibrated confidence.")

RELIABILITY REPORT
----------------------------------------
1. Investigate incorrect predictions made with high confidence.

Important: high confidence is not automatically calibrated confidence.


In [4]:


%%writefile app.py
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import numpy as np

app = FastAPI(
    title="AI Model Reliability Lab",
    description="ML metrics, confidence-error analysis, and reliability warnings",
    version="1.0.0"
)

class EvaluationInput(BaseModel):
    true_labels: List[int] = Field(..., min_length=1)
    predicted_labels: List[int] = Field(..., min_length=1)
    confidences: Optional[List[float]] = None

@app.get("/")
def home():
    return {
        "project": "AI Model Reliability Lab",
        "status": "running",
        "documentation": "/docs"
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/evaluate")
def evaluate(data: EvaluationInput):
    true = data.true_labels
    pred = data.predicted_labels
    conf = data.confidences

    if len(true) != len(pred):
        raise HTTPException(422, "Label lists must have equal lengths.")

    if any(x not in (0, 1) for x in true + pred):
        raise HTTPException(422, "Labels must be 0 or 1.")

    if conf is not None:
        if len(conf) != len(true):
            raise HTTPException(422, "Confidence count must match labels.")
        if any(not 0 <= c <= 1 for c in conf):
            raise HTTPException(422, "Confidence must be between 0 and 1.")

    errors = [i for i, (t, p) in enumerate(zip(true, pred)) if t != p]

    result = {
        "sample_count": len(true),
        "accuracy": round(accuracy_score(true, pred), 4),
        "precision": round(precision_score(
            true, pred, zero_division=0
        ), 4),
        "recall": round(recall_score(
            true, pred, zero_division=0
        ), 4),
        "f1_score": round(f1_score(
            true, pred, zero_division=0
        ), 4),
        "confusion_matrix": confusion_matrix(
            true, pred, labels=[0, 1]
        ).tolist(),
        "error_count": len(errors),
        "recommendations": []
    }

    if result["recall"] < 0.8:
        result["recommendations"].append(
            "Review false negatives and decision thresholds."
        )

    if result["precision"] < 0.8:
        result["recommendations"].append(
            "Review false positives and feature quality."
        )

    if conf is not None:
        flagged = [i for i in errors if conf[i] >= 0.8]
        result["confidence_analysis"] = {
            "mean_confidence": round(float(np.mean(conf)), 4),
            "high_confidence_wrong_count": len(flagged),
            "high_confidence_wrong_indices_zero_based": flagged
        }

        if flagged:
            result["recommendations"].append(
                "Review incorrect predictions with confidence >= 80%."
            )

    if not result["recommendations"]:
        result["recommendations"].append(
            "No basic metric warning triggered; test on independent data."
        )

    return result

Writing app.py


In [5]:


from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

# Test 1: health endpoint
response = client.get("/health")
assert response.status_code == 200
assert response.json()["status"] == "ok"
print("PASS: health check")

# Test 2: metric evaluation
response = client.post("/evaluate", json={
    "true_labels": [0, 1, 1, 0],
    "predicted_labels": [0, 1, 0, 0],
    "confidences": [0.91, 0.95, 0.88, 0.79]
})
assert response.status_code == 200
result = response.json()
assert result["sample_count"] == 4
assert result["error_count"] == 1
assert result["confidence_analysis"]["high_confidence_wrong_count"] == 1
print("PASS: metrics and confidence analysis")
print(result)

# Test 3: reject invalid labels
response = client.post("/evaluate", json={
    "true_labels": [0, 2],
    "predicted_labels": [0, 1]
})
assert response.status_code == 422
print("PASS: input validation")

print("\nAll three API tests passed.")

PASS: health check
PASS: metrics and confidence analysis
{'sample_count': 4, 'accuracy': 0.75, 'precision': 1.0, 'recall': 0.5, 'f1_score': 0.6667, 'confusion_matrix': [[2, 0], [1, 1]], 'error_count': 1, 'recommendations': ['Review false negatives and decision thresholds.', 'Review incorrect predictions with confidence >= 80%.'], 'confidence_analysis': {'mean_confidence': 0.8825, 'high_confidence_wrong_count': 1, 'high_confidence_wrong_indices_zero_based': [2]}}
PASS: input validation

All three API tests passed.


In [6]:


from pathlib import Path

project = Path("/content/ai-model-reliability-lab")
project.mkdir(exist_ok=True)

(project / "app.py").write_text(
    Path("/content/app.py").read_text()
)

(project / "requirements.txt").write_text(
    "fastapi>=0.110,<1.0\n"
    "uvicorn>=0.27,<1.0\n"
    "scikit-learn>=1.4,<2.0\n"
    "numpy>=1.24,<3.0\n"
    "httpx>=0.26,<1.0\n"
    "pytest>=8,<9\n"
)

(project / "README.md").write_text("""# AI Model Reliability Lab

A lightweight ML evaluation and reliability API.

## Features
- Accuracy, precision, recall, and F1-score
- Confusion matrix and error counting
- High-confidence incorrect prediction flags
- Input validation
- FastAPI interactive documentation
- Automated tests

## Run
pip install -r requirements.txt
uvicorn app:app --host 0.0.0.0 --port 7860

Open /docs for the API interface and /health for the health check.

## Limitations
This is an educational benchmark. Confidence warnings do not prove
probability calibration. Results do not establish real-world
generalization, fairness, or clinical suitability.
""")

(project / ".gitignore").write_text(
    "__pycache__/\n.pytest_cache/\n.env\n.venv/\n"
)

print("Project files:")
for f in sorted(project.iterdir()):
    print(f.name)

Project files:
.gitignore
README.md
app.py
requirements.txt


In [7]:


import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/ai-model-reliability-lab",
    "zip",
    root_dir="/content/ai-model-reliability-lab"
)

print("ZIP ready:", zip_path)
files.download(zip_path)

ZIP ready: /content/ai-model-reliability-lab.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>